In [1]:
#import librarys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn import linear_model, datasets
from sklearn.model_selection import GridSearchCV

import seaborn as sns
#for feature transformation
from sklearn.preprocessing import StandardScaler

#for random forest
from sklearn.ensemble import RandomForestRegressor

#for plot
import matplotlib.pyplot as plt

#for decision tree accuracy score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.cluster import KMeans

#for decision tree 
from sklearn.tree import DecisionTreeClassifier

#for feature selection
from sklearn.feature_selection import SelectKBest, chi2

#for gridsearch 
from sklearn.model_selection import GridSearchCV

#for regression 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#for LGMR regression
from sklearn.feature_selection import VarianceThreshold
from lightgbm import LGBMRegressor

The Titanic Dataset 

The analysis will use the well-known Titanic dataset, which includes 12 variables and 1308 rows in the dataset. Passenger ID, survival (binary: 0=No, 1=Yes), pclass (Ticket class 1=1st, 2=2nd, 3=3rd), name, sex, age, sibsp (number of siblings + spouses aboard the Titanic), parch (number of parents + children aboard the Titanic), ticket (number), fare , cabin (number), embarked (Port of Embarkation: C=Cherbourg, Q=Queenstown, S=Southampton). The dataset was provided pre-split into training and test datasets for model creation and testing on unseen data.

In [2]:
#read data files

url_1 = "https://raw.githubusercontent.com/Jess-O45/GROUP_PROJECT/main/test.csv"
test = pd.read_csv(url_1, encoding="latin1")  # or utf-8 if latin1 fails

#test.head()

In [3]:
url_2 = "https://raw.githubusercontent.com/Jess-O45/GROUP_PROJECT/main/train.csv"
train = pd.read_csv(url_2, encoding="latin1")  # or utf-8 if latin1 fails


In [4]:
#Understanding the test data
print(test.shape)        # (rows, columns)
print(test.dtypes)       # column data types
print(test.info())

(418, 11)
PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB
None


In [5]:
##Clean Test data

In [6]:
test[test.isnull().any(axis=1)]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
412,1304,3,"Henriksson, Miss. Jenny Lovisa",female,28.0,0,0,347086,7.7750,NaN,S
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [7]:
#fill in NaN for fare based on Pclass

avg_fare = test.groupby('Pclass')['Fare'].mean()
print(avg_fare)

Pclass
1    94.280297
2    22.202104
3    12.459678
Name: Fare, dtype: float64


In [8]:
# Map Pclass to its average fare and fill missing Fare
test['Fare'] = test['Fare'].fillna(test['Pclass'].map(avg_fare))

In [9]:
#fill in age NaNs - Group children and adults separately based on 1/0 of Parch (parent on board) 

# Create flag: 1 = child (has parent), 0 = adult
test['ChildFlag'] = (test['Parch'] > 0).astype(int)

avg_age = test.groupby('ChildFlag')['Age'].mean()
print(avg_age)

def fill_age(row):
    if pd.isna(row['Age']):
        return avg_age[row['ChildFlag']]
    return row['Age']

test['Age'] = test.apply(fill_age, axis=1)

ChildFlag
0    31.528340
1    26.623529
Name: Age, dtype: float64


In [22]:
#group cabins by fare range and pclass

test['FareBin'] = pd.qcut(test['Fare'], 3)  # bins

pivot = test.pivot_table(
    index=['FareBin', 'Pclass'],
    columns='Cabin',
    values='Fare',
    aggfunc='count',
    fill_value=0,
    observed=False)

pivot

Cabin                   A11  A18  A21  A29  A34  A9  B10  B11  B24  B26  ...  \
FareBin         Pclass                                                   ...   
(-0.001, 8.662] 1         0    0    0    0    0   0    0    0    0    0  ...   
                2         0    0    0    0    0   0    0    0    0    0  ...   
                3         0    0    0    0    0   0    0    0    0    0  ...   
(8.662, 26.0]   1         5    0    0    0    0   0    0    0    0    0  ...   
                2         0    0    0    0    0   0    0    0    0    0  ...   
                3         0    0    0    0    0   0    0    0    0    0  ...   
(26.0, 512.329] 1         8    1    1    1    2   1    1    1    1    1  ...   
                2         0    0    0    0    0   0    0    0    0    0  ...   
                3         0    0    0    0    0   0    0    0    0    0  ...   

Cabin                   E52  E60  F  F E46  F E57  F G63  F2  F33  F4  G6  
FareBin         Pclass                                                     
(-0.001, 8.662] 1         0    0  0      0      0      0   0    0   0   0  
                2         0    0  0      0      0      0   0    0   0   0  
                3         0    0  0    110      1      1   0    0   0   0  
(8.662, 26.0]   1         0    1  0      0      0      0   0    0   0   0  
                2         0    0  1      0      0      0   1    1   0   0  
                3         0    0  0      0      0      0   0    0   0  38  
(26.0, 512.329] 1         1    0  0      0      0      0   0    0   0   0  
                2         0    0  0      0      0      0   0    0  17   0  
                3         0    0  0      0      0      0   0    0   0   4  

[9 rows x 76 columns]

In [23]:
#map FareBin and Pclass to fill in Cabin
mode_full_lookup = (
    test.dropna(subset=['Cabin'])
        .groupby(['FareBin','Pclass'], observed=False)['Cabin']
        .agg(lambda x: x.mode().iat[0]))


In [24]:
#Fill in the cabin based on the FAREBIN and Pclass to replace NaN in Cabin 

def fill_cabin(row):
    if pd.isna(row['Cabin']):
        key = (row['FareBin'], row['Pclass'])
        if key in mode_full_lookup.index:
            return mode_full_lookup.loc[key]   # use full real cabin
        else:
            return "UNK000"                    # fallback if no data
    return row['Cabin']

test['Cabin'] = test.apply(fill_cabin, axis=1)


In [25]:
test[test.isnull().any(axis=1)] #no nulls

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,ChildFlag,FareBin


In [14]:
##Clean Train data

In [15]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [16]:
#Understanding the train data
print(train.shape)        # (rows, columns)
print(train.dtypes)       # column data types
print(train.info())

(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes:

In [17]:
train[train.isnull().any(axis=1)]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.0500,NaN,S
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S


In [ ]:
#need to fill in NaN Cabin. 

In [18]:
#need to fill in NaN embarked. 


In [ ]:
##Notate when all data is clean

In [20]:
#Make histograms to understand distributions for numeric data



In [21]:
#Make bar charts to understand balance of classes for categorical data

For milestone 4

Explain your process for prepping the data
Build and evaluate at least one model
Interpret your results
Begin to formulate a conclusion/recommendations